In [1]:
from pydantic import BaseModel, Field
from typing import List, Literal


class ReviewSummary(BaseModel):
    summary: str = Field(description="Short summary of the review")
    topic: str = Field(description="Main topic discussed")
    pros: List[str] = Field(description="Positive points")
    cons: List[str] = Field(description="Negative points")
    sentiment: Literal["positive", "negative", "neutral", "mixed"]

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3)
structured_llm = model.with_structured_output(ReviewSummary)

In [3]:
reviews = [
    """
    I have been using this laptop for a little over three months, primarily for software development,
    machine learning experiments, and general productivity tasks. The overall performance has been
    excellent. The system boots in under ten seconds, applications launch almost instantly, and
    multitasking feels smooth even when I have multiple browser windows, VS Code, Docker containers,
    and Jupyter notebooks running simultaneously.

    The display is bright, color accurate, and comfortable for long coding sessions. The keyboard
    offers good key travel and feels satisfying to type on. Battery life has consistently lasted
    between eight and ten hours during normal usage, which is impressive considering the workload.
    The build quality also feels premium, with very little flex in the chassis.

    However, the laptop is not perfect. The speakers are disappointing and lack depth, making movies
    and music less enjoyable without headphones. The cooling system can become noticeably loud during
    heavy workloads such as model training or video rendering. I have also noticed that the webcam
    quality is mediocre and struggles in low-light environments.

    Despite these shortcomings, the combination of performance, battery life, display quality, and
    portability makes this one of the best laptops I have owned. I would confidently recommend it to
    students, developers, and professionals who need a reliable machine for demanding daily work.
    """,

    """
    I purchased this smartphone approximately four months ago and have used it extensively for
    photography, social media, video streaming, navigation, and communication. The camera system is
    undoubtedly the highlight of the device. Photos are sharp, colors are vibrant, and low-light
    performance is significantly better than my previous phone. The display is beautiful, offering
    excellent brightness and smooth scrolling thanks to the high refresh rate.

    Battery life has generally been good, lasting an entire day even with moderate to heavy usage.
    Charging speeds are fast enough that I rarely worry about running out of power. The software
    experience is clean and responsive, with very few bugs or crashes encountered so far.

    On the negative side, the device is more expensive than many competitors offering similar
    specifications. While the build quality is premium, the glass back attracts fingerprints very
    easily and requires frequent cleaning. The included storage feels somewhat limited for users who
    capture a large number of photos and videos. Additionally, the phone can become slightly warm
    during gaming sessions or extended camera usage.

    Overall, I am satisfied with the purchase. The excellent camera, display quality, and smooth
    software experience outweigh the drawbacks. While it may not offer the best value for money, it
    delivers a premium user experience that many customers will appreciate.
    """,

    """
    My experience with this online learning platform has been mixed. The platform contains a large
    library of courses covering programming, data science, artificial intelligence, cloud computing,
    and business skills. The instructors are generally knowledgeable and explain concepts clearly.
    The video quality is high, and the mobile application makes it easy to continue learning while
    traveling.

    One feature I particularly appreciate is the hands-on project approach. Many courses include
    practical assignments that reinforce the concepts being taught. Progress tracking and completion
    certificates are also useful for maintaining motivation and demonstrating learning achievements.

    Unfortunately, there are several areas where the platform could improve. Course quality varies
    significantly between instructors, making it difficult to know whether a course will meet
    expectations before starting. Some content feels outdated, especially in rapidly evolving fields
    such as machine learning and generative AI. Customer support response times have also been slower
    than expected, with some issues taking several days to resolve.

    Pricing is reasonable compared to traditional education, but certain premium features are locked
    behind higher subscription tiers. Overall, the platform provides strong educational value and is
    worth considering for self-directed learners, though improvements in content consistency and
    support would make the experience substantially better.
    """,

    """
    I recently purchased this wireless headphone set for daily commuting, remote work meetings, and
    casual music listening. The audio quality is excellent, with clear vocals, strong bass response,
    and a balanced sound profile suitable for multiple genres of music. Noise cancellation performs
    surprisingly well, especially in crowded public transport environments where background noise is
    significantly reduced.

    Comfort is another major strength. I have worn the headphones for several hours at a time without
    experiencing discomfort. The battery life exceeds the manufacturer's claims in my experience,
    often lasting multiple days before requiring a recharge. Bluetooth connectivity is stable and
    pairing with multiple devices is straightforward.

    There are some weaknesses worth mentioning. The carrying case feels cheaper than expected given
    the product's premium positioning. Touch controls occasionally fail to register inputs correctly,
    resulting in accidental pauses or skipped tracks. The companion mobile application is functional
    but lacks advanced customization features available from competing brands.

    Even with these issues, the overall value remains strong. Excellent sound quality, effective
    noise cancellation, and long battery life make these headphones an easy recommendation for users
    seeking a reliable wireless audio solution for both work and entertainment purposes.
    """,

    """
    Our company adopted this project management software approximately six months ago to coordinate
    development teams, marketing campaigns, and cross-functional initiatives. The platform has
    improved visibility into project progress and made collaboration substantially easier. Task
    assignments, deadline tracking, and workflow automation features have helped reduce manual work
    and improve accountability across teams.

    The user interface is modern and relatively intuitive. New employees are able to become
    productive within a short period of time. Integration with communication tools, cloud storage
    services, and calendar applications has streamlined many daily processes. Reporting dashboards
    provide useful insights into team productivity and project performance.

    However, the software is not without challenges. Some advanced features require extensive setup
    and configuration before they become useful. Performance occasionally slows when dealing with
    very large projects containing thousands of tasks. Pricing can also become expensive as team size
    grows, making it less attractive for smaller organizations with limited budgets.

    Customer support has generally been helpful, although response times vary depending on the
    complexity of the issue. Overall, the software delivers meaningful productivity improvements and
    has become an important part of our workflow. Despite a few performance and pricing concerns, the
    benefits significantly outweigh the drawbacks for medium to large teams.
    """
]

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Analyze the review and extract:

            - summary
            - topic
            - pros
            - cons
            - sentiment

            Return structured data only.
            """
        ),
        ("human", "{review}")
    ]
)

chain = prompt | structured_llm

In [5]:
results = chain.batch(
    [{"review": review} for review in reviews]
)

for result in results:
    print(result)

summary='A high-performance laptop excellent for development and productivity, though hampered by poor audio and webcam quality.' topic='Laptop performance and usability' pros=['Excellent performance for multitasking and development', 'Fast boot and application launch times', 'Bright and color-accurate display', 'Satisfying keyboard feel', 'Impressive battery life', 'Premium build quality'] cons=['Disappointing speaker quality', 'Loud cooling system under heavy load', 'Mediocre webcam performance in low light'] sentiment='positive'
summary='A high-performing smartphone with an exceptional camera and display, though it comes at a premium price and has some minor thermal and storage limitations.' topic='Smartphone review' pros=['Excellent camera system with vibrant colors and good low-light performance', 'Beautiful, bright display with high refresh rate', 'Reliable battery life', 'Fast charging speeds', 'Clean and responsive software'] cons=['Expensive compared to competitors', 'Glass ba

In [6]:
for i, result in enumerate(results, start=1):
    print(f"\n{'=' * 80}")
    print(f"REVIEW {i}")
    print(f"{'=' * 80}")

    print(f"📌 Topic     : {result.topic}") # type: ignore
    print(f"😊 Sentiment : {result.sentiment}") # type: ignore

    print("\n📝 Summary")
    print(result.summary)# type: ignore

    print("\n✅ Pros")
    for pro in result.pros: # type: ignore
        print(f"  • {pro}")

    print("\n❌ Cons")
    for con in result.cons:# type: ignore
        print(f"  • {con}")

    print()


REVIEW 1
📌 Topic     : Laptop performance and usability
😊 Sentiment : positive

📝 Summary
A high-performance laptop excellent for development and productivity, though hampered by poor audio and webcam quality.

✅ Pros
  • Excellent performance for multitasking and development
  • Fast boot and application launch times
  • Bright and color-accurate display
  • Satisfying keyboard feel
  • Impressive battery life
  • Premium build quality

❌ Cons
  • Disappointing speaker quality
  • Loud cooling system under heavy load
  • Mediocre webcam performance in low light


REVIEW 2
📌 Topic     : Smartphone review
😊 Sentiment : positive

📝 Summary
A high-performing smartphone with an exceptional camera and display, though it comes at a premium price and has some minor thermal and storage limitations.

✅ Pros
  • Excellent camera system with vibrant colors and good low-light performance
  • Beautiful, bright display with high refresh rate
  • Reliable battery life
  • Fast charging speeds
  • Cl

In [7]:
import json

for result in results:
    print(
        json.dumps(
            result.model_dump(),
            indent=4,
            ensure_ascii=False
        )
    )

{
    "summary": "A high-performance laptop excellent for development and productivity, though hampered by poor audio and webcam quality.",
    "topic": "Laptop performance and usability",
    "pros": [
        "Excellent performance for multitasking and development",
        "Fast boot and application launch times",
        "Bright and color-accurate display",
        "Satisfying keyboard feel",
        "Impressive battery life",
        "Premium build quality"
    ],
    "cons": [
        "Disappointing speaker quality",
        "Loud cooling system under heavy load",
        "Mediocre webcam performance in low light"
    ],
    "sentiment": "positive"
}
{
    "summary": "A high-performing smartphone with an exceptional camera and display, though it comes at a premium price and has some minor thermal and storage limitations.",
    "topic": "Smartphone review",
    "pros": [
        "Excellent camera system with vibrant colors and good low-light performance",
        "Beautiful, brigh

In [8]:
from rich.console import Console
from rich.panel import Panel

console = Console()

for i, result in enumerate(results, 1):

    content = f"""
[bold]Summary:[/bold]
{result.summary } 

[bold green]Pros:[/bold green]
""" + "\n".join(f"• {p}" for p in result.pros)# type: ignore

    content += "\n\n[bold red]Cons:[/bold red]\n"
    content += "\n".join(f"• {c}" for c in result.cons)# type: ignore

    console.print(
        Panel(
            content,
            title=f"{i}. {result.topic} ({result.sentiment})"# type: ignore
        )
    )

╭──────────────────────────────── 1. Laptop performance and usability (positive) ─────────────────────────────────╮
│                                                                                                                 │
│ Summary:                                                                                                        │
│ A high-performance laptop excellent for development and productivity, though hampered by poor audio and webcam  │
│ quality.                                                                                                        │
│                                                                                                                 │
│ Pros:                                                                                                           │
│ • Excellent performance for multitasking and development                                                        │
│ • Fast boot and application launch times                                                                        │
│ • Bright and color-accurate display                                                                             │
│ • Satisfying keyboard feel                                                                                      │
│ • Impressive battery life                                                                                       │
│ • Premium build quality                                                                                         │
│                                                                                                                 │
│ Cons:                                                                                                           │
│ • Disappointing speaker quality                                                                                 │
│ • Loud cooling system under heavy load                                                                          │
│ • Mediocre webcam performance in low light                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 2. Smartphone review (positive) ────────────────────────────────────────╮
│                                                                                                                 │
│ Summary:                                                                                                        │
│ A high-performing smartphone with an exceptional camera and display, though it comes at a premium price and has │
│ some minor thermal and storage limitations.                                                                     │
│                                                                                                                 │
│ Pros:                                                                                                           │
│ • Excellent camera system with vibrant colors and good low-light performance                                    │
│ • Beautiful, bright display with high refresh rate                                                              │
│ • Reliable battery life                                                                                         │
│ • Fast charging speeds                                                                                          │
│ • Clean and responsive software                                                                                 │
│                                                                                                                 │
│ Cons:                                                                                                           │
│ • Expensive compared to competitors                                                                             │
│ • Glass back attracts fingerprints                                                                              │
│ • Limited storage capacity                                                                                      │
│ • Device warms up during intensive tasks like gaming or long camera use                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── 3. Online learning platform (mixed) ──────────────────────────────────────╮
│                                                                                                                 │
│ Summary:                                                                                                        │
│ The platform offers a vast library and practical learning tools, but suffers from inconsistent course quality   │
│ and slow customer support.                                                                                      │
│                                                                                                                 │
│ Pros:                                                                                                           │
│ • Large library of diverse courses                                                                              │
│ • Knowledgeable instructors                                                                                     │
│ • High video quality                                                                                            │
│ • Convenient mobile application                                                                                 │
│ • Hands-on project approach                                                                                     │
│ • Useful progress tracking and certificates                                                                     │
│ • Reasonable pricing                                                                                            │
│                                                                                                                 │
│ Cons:                                                                                                           │
│ • Inconsistent course quality                                                                                   │
│ • Outdated content in some technical fields                                                                     │
│ • Slow customer support response times                                                                          │
│ • Premium features locked behind higher tiers                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── 4. Wireless headphones (positive) ───────────────────────────────────────╮
│                                                                                                                 │
│ Summary:                                                                                                        │
│ High-quality wireless headphones with excellent sound and battery life, though hampered by minor control and    │
│ software issues.                                                                                                │
│                                                                                                                 │
│ Pros:                                                                                                           │
│ • Excellent audio quality                                                                                       │
│ • Effective noise cancellation                                                                                  │
│ • Comfortable for long-term wear                                                                                │
│ • Long battery life                                                                                             │
│ • Stable Bluetooth connectivity                                                                                 │
│                                                                                                                 │
│ Cons:                                                                                                           │
│ • Cheap feeling carrying case                                                                                   │
│ • Unreliable touch controls                                                                                     │
│ • Limited companion app features                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── 5. Project management software (positive) ───────────────────────────────────╮
│                                                                                                                 │
│ Summary:                                                                                                        │
│ The project management software significantly improves team productivity and collaboration, though it has some  │
│ performance limitations and high costs for scaling.                                                             │
│                                                                                                                 │
│ Pros:                                                                                                           │
│ • Improved project visibility                                                                                   │
│ • Enhanced collaboration                                                                                        │
│ • Effective workflow automation                                                                                 │
│ • Modern and intuitive user interface                                                                           │
│ • Strong integration capabilities                                                                               │
│ • Useful reporting dashboards                                                                                   │
│                                                                                                                 │
│ Cons:                                                                                                           │
│ • Complex setup for advanced features                                                                           │
│ • Performance slowdowns with large projects                                                                     │
│ • Expensive pricing for growing teams                                                                           │
│ • Inconsistent customer support response times                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯